# CNM Analysis: pA LHC (8.16 TeV)

**System:** pPb 8.16 TeV  
**Physics:** nPDF (EPPS21) × Energy Loss (Arleo-Peigne) × pT Broadening (Cronin)  
**Module:** `cnm_combine_fast` (Vectorized)

This notebook generates the coherent energy loss + broadening corrections combined with nPDF effects.

In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add paths
HERE = Path(".").resolve()
sys.path.append(str(HERE.parent / "eloss_code"))
sys.path.append(str(HERE.parent / "npdf_code"))
sys.path.append(str(HERE.parent / "cnm_combine"))

from cnm_combine_fast import CNMCombineFast

# Plotting style
plt.style.use('seaborn-v0_8-paper')
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12

OUTDIR = HERE / "outputs" / "calculated_cnm_results"
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"Saving outputs to {OUTDIR}")

Saving outputs to /home/sawin/Desktop/Charmonia/charmonia_combined_analysis/cnm_notebook/outputs/calculated_cnm_results


## 1. Initialize CNM Combine (LHC)

In [2]:
cnm = CNMCombineFast.from_defaults(
    energy="8.16",
    family="charmonia",
    particle_state="avg"
)

print(f"Initialized {cnm.system} @ {cnm.energy} TeV")
print(f"Centrality bins: {cnm.cent_bins}")

[OpticalGlauber] Target A=208, d=0.549 fm, σ_nn=71.00 mb
[OpticalGlauber] ∫ d²s T_A(s) ≈ 208.483  (should be ~A=208)
[OpticalGlauber] ∫ d²s T_d(s) ≈ 1.9972  (should be ~2)
[OpticalGlauber] Tabulating overlaps T_AA(b), T_pA(b), T_dA(b)...
[OpticalGlauber] σ_tot: AA=7757.8 mb, pA=1925.9 mb, dA=2381.9 mb


AttributeError: 'CNMCombineFast' object has no attribute 'system'

## 2. RpA vs Rapidity (pT integrated)

In [ ]:
%%time
y_cent, labels, bands_y = cnm.cnm_vs_y()

print("Computation done.")

In [ ]:
# Plotting Helper
def plot_bands(x, bands, labels, xlabel, ylabel="$R_{pA}$", title=""):
    fig, ax = plt.subplots(figsize=(9, 6))
    
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(labels)))
    color_map = {lab: col for lab, col in zip(labels, colors)}
    color_map["MB"] = "black"
    
    # Order: Most central to most peripheral, then MB
    plot_order = list(labels) + ["MB"]
    
    comp_styles = {
        "cnm": ("-", 0.2),
        "npdf": ("--", 0.1),
        "eloss_broad": (":", 0.1)
    }
    
    for comp, (ls, alpha) in comp_styles.items():
        if comp not in bands: continue
        Dc, Dlo, Dhi = bands[comp]
        
        for lab in plot_order:
            c = color_map.get(lab, "gray")
            if comp == "cnm":
                # Main band
                ax.plot(x, Dc[lab], color=c, linestyle=ls, label=f"{lab}" if comp=="cnm" else None, linewidth=2)
                ax.fill_between(x, Dlo[lab], Dhi[lab], color=c, alpha=alpha)
            else:
                # Components: thinner, no legend
                ax.plot(x, Dc[lab], color=c, linestyle=ls, linewidth=1, alpha=0.7)
                # ax.fill_between(x, Dlo[lab], Dhi[lab], color=c, alpha=alpha*0.5)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0.0, 1.4)
    ax.set_title(title)
    ax.axhline(1.0, color='k', linewidth=0.5)
    
    # Custom legend for components
    from matplotlib.lines import Line2D
    custom_lines = [
        Line2D([0], [0], color='k', lw=2, linestyle='-'),
        Line2D([0], [0], color='k', lw=1, linestyle='--'),
        Line2D([0], [0], color='k', lw=1, linestyle=':')
    ]
    leg1 = ax.legend(custom_lines, ['Total CNM', 'nPDF only', 'Eloss+Broad only'], loc='lower left', fontsize='small')
    ax.add_artist(leg1)
    
    # Legend for centrality
    ax.legend(loc='upper right', ncol=2)
    
    return fig

fig = plot_bands(y_cent, bands_y, labels, "$y_{CMS}$", title="LHC pPb 8.16 TeV: $R_{pA}$ vs $y$")
fig.savefig(OUTDIR / "05_d_LHC_RpA_vs_y.pdf")
fig.savefig(OUTDIR / "05_d_LHC_RpA_vs_y.png", dpi=150)
print("Saved plot.")

## 3. RpA vs pT (in rapidity windows)

In [ ]:
windows = cnm.y_windows
print("Analyzing windows:", windows)

for (y0, y1, desc) in windows:
    print(f"Processing {desc}...")
    pt_cent, labels, bands_pt = cnm.cnm_vs_pT(y_window=(y0, y1))
    
    fig = plot_bands(pt_cent, bands_pt, labels, "$p_T$ [GeV]", title=f"LHC pPb: {desc}")
    safe_desc = desc.replace(" ", "_").replace("<", "lt").replace(">", "gt")
    fig.savefig(OUTDIR / f"05_d_LHC_RpA_vs_pT_{y0}_{y1}.png", dpi=150)
    plt.close(fig)

## 4. RpA vs Centrality

In [ ]:
# Plotting Helper for Centrality
def plot_cent_bands(cnm_cent_data, labels, title=""):
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # x-axis: Npart roughly? Or just bin index?
    # We map bin labels to Npart mean or just place them equidistantly
    # For simplicity, use 0...N
    x = np.arange(len(labels))
    
    # Component styles
    comp_styles = {
        "cnm": ("blue", "-", "Total CNM"),
        "npdf": ("green", "--", "nPDF"),
        "eloss_broad": ("red", ":", "Eloss+Broad")
    }
    
    for comp, (col, ls, name) in comp_styles.items():
        if comp not in cnm_cent_data: continue
        
        (Vc, Vlo, Vhi, MBc, MBlo, MBhi) = cnm_cent_data[comp]
        
        # Plot central points
        ax.errorbar(x, Vc, yerr=[Vc-Vlo, Vhi-Vc], fmt='o', color=col, linestyle=ls, label=name, capsize=4)
        
        # MB band as horizontal strip
        ax.axhspan(MBlo, MBhi, color=col, alpha=0.1)
        ax.axhline(MBc, color=col, linestyle=ls, linewidth=0.5)
    
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel("Centrality Bin")
    ax.set_ylabel("$R_{pA}$")
    ax.set_title(title)
    ax.set_ylim(0.0, 1.3)
    ax.legend()
    return fig

for (y0, y1, desc) in windows:
    res = cnm.cnm_vs_centrality(y_window=(y0, y1))
    fig = plot_cent_bands(res, labels, title=f"LHC pPb Centrality: {desc}")
    fig.savefig(OUTDIR / f"05_d_LHC_RpA_vs_cent_{y0}_{y1}.png", dpi=150)
    plt.close(fig)